In [27]:
# Cell 1 - Install dependencies
import sys
!{sys.executable} -m pip install chromadb sentence-transformers tqdm kagglehub -q

You should consider upgrading via the '/Users/jicijiang/neu/cs6200/project/.cs6200/bin/python -m pip install --upgrade pip' command.


In [28]:
# Cell 2 - Detect environment and set paths
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").exists()
print(f"Environment: {'Kaggle' if ON_KAGGLE else 'Local'}")

if ON_KAGGLE:
    # Clone repo to get queries/queries.json
    import subprocess
    subprocess.run(
        ["git", "clone", "https://github.com/YanBoChen0928/FairSearch-qBio.git",
         "/kaggle/working/FairSearch-qBio", "--depth", "1", "-q"],
        check=True
    )
    QUERIES_PATH = Path("/kaggle/working/FairSearch-qBio/queries/queries.json")
    CHROMA_SRC   = Path("/kaggle/input/fairsearch-qbio-chromadb/chroma_db")
    CHROMA_PATH  = Path("/kaggle/working/chroma_db")
    OUT_PATH     = Path("/kaggle/working/retrieval_results.json")
else:
    # Locally: queries.json is already in the repo
    REPO_ROOT = next(
        c.resolve() for c in [Path("."), Path("..")]
        if (c / "queries" / "queries.json").exists()
    )
    QUERIES_PATH = REPO_ROOT / "queries" / "queries.json"
    CHROMA_SRC   = None  # will be downloaded via kagglehub in Cell 3
    CHROMA_PATH  = REPO_ROOT / "data" / "chroma"
    OUT_PATH     = REPO_ROOT / "data" / "processed" / "retrieval_results.json"

print(f"QUERIES_PATH : {QUERIES_PATH}")
print(f"CHROMA_PATH  : {CHROMA_PATH}")
print(f"OUT_PATH     : {OUT_PATH}")

Environment: Local
QUERIES_PATH : /Users/jicijiang/neu/cs6200/project/FairSearch-qBio/queries/queries.json
CHROMA_PATH  : /Users/jicijiang/neu/cs6200/project/FairSearch-qBio/data/chroma
OUT_PATH     : /Users/jicijiang/neu/cs6200/project/FairSearch-qBio/data/processed/retrieval_results.json


In [29]:
# Cell 3 - Set up ChromaDB
import shutil
import chromadb

if ON_KAGGLE:
    # Copy from read-only Kaggle input to writable working directory
    if CHROMA_PATH.exists():
        shutil.rmtree(CHROMA_PATH)
    shutil.copytree(CHROMA_SRC, CHROMA_PATH)
    print(f"Copied ChromaDB from Kaggle input to {CHROMA_PATH}")
else:
    # Download via kagglehub if not already present
    chroma_sqlite = CHROMA_PATH / "chroma.sqlite3"
    if chroma_sqlite.exists():
        print(f"ChromaDB already exists at {CHROMA_PATH} — skipping download.")
    else:
        import kagglehub
        dl_path = Path(kagglehub.dataset_download("rajlucka/fairsearch-qbio-chromadb"))
        # Locate chroma.sqlite3 inside the downloaded folder
        candidates = list(dl_path.rglob("chroma.sqlite3"))
        if not candidates:
            raise FileNotFoundError("chroma.sqlite3 not found in downloaded dataset.")
        src = candidates[0].parent
        CHROMA_PATH.mkdir(parents=True, exist_ok=True)
        for item in src.rglob("*"):
            if item.name == ".gitkeep":
                continue
            rel = item.relative_to(src)
            dst = CHROMA_PATH / rel
            if item.is_dir():
                dst.mkdir(parents=True, exist_ok=True)
            else:
                dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(item, dst)
        print(f"Downloaded and copied ChromaDB to {CHROMA_PATH}")

# Connect and verify
client     = chromadb.PersistentClient(path=str(CHROMA_PATH))
collection = client.get_collection("qbio_papers")
print(f"Collection 'qbio_papers': {collection.count():,} records")

Downloaded and copied ChromaDB to /Users/jicijiang/neu/cs6200/project/FairSearch-qBio/data/chroma
Collection 'qbio_papers': 55,300 records


In [30]:
# Cell 4 - Load queries
import json

with open(QUERIES_PATH, "r") as f:
    queries = json.load(f)

print(f"Loaded {len(queries)} queries.")
print(f"Example: [{queries[0]['query_id']}] ({queries[0]['subcategory']}) {queries[0]['query_text'][:70]}")

Loaded 50 queries.
Example: [q001] (q-bio.QM) How can stochastic differential equations model gene expression noise 


In [31]:
# Cell 5 - Pre-compute relevant pool size per subcategory (Recall@10 denominator)
# Fetches metadata in batches of 5000 to avoid SQLite "too many SQL variables" error

target_subcats = sorted(set(q["subcategory"] for q in queries))
print(f"Subcategories: {target_subcats}\n")

BATCH_SIZE = 5000
total = collection.count()
subcat_pool_size = {sc: 0 for sc in target_subcats}

print(f"Scanning {total:,} records in batches of {BATCH_SIZE}...")
for offset in range(0, total, BATCH_SIZE):
    batch = collection.get(limit=BATCH_SIZE, offset=offset, include=["metadatas"])
    for meta in batch["metadatas"]:
        cats = meta.get("categories", "")
        for sc in target_subcats:
            if sc in cats:
                subcat_pool_size[sc] += 1
    print(f"  processed {min(offset + BATCH_SIZE, total):,} / {total:,}", end="\r")

print("\nDone.\n")
for sc in sorted(subcat_pool_size):
    print(f"  {sc}: {subcat_pool_size[sc]:,} papers in corpus")
print(f"\nPool sizes computed for {len(subcat_pool_size)} subcategories.")

Subcategories: ['q-bio.BM', 'q-bio.CB', 'q-bio.GN', 'q-bio.MN', 'q-bio.NC', 'q-bio.OT', 'q-bio.PE', 'q-bio.QM', 'q-bio.SC', 'q-bio.TO']

Scanning 55,300 records in batches of 5000...
  processed 55,300 / 55,300
Done.

  q-bio.BM: 6,745 papers in corpus
  q-bio.CB: 2,451 papers in corpus
  q-bio.GN: 3,835 papers in corpus
  q-bio.MN: 4,157 papers in corpus
  q-bio.NC: 12,131 papers in corpus
  q-bio.OT: 1,586 papers in corpus
  q-bio.PE: 12,998 papers in corpus
  q-bio.QM: 13,180 papers in corpus
  q-bio.SC: 1,808 papers in corpus
  q-bio.TO: 2,618 papers in corpus

Pool sizes computed for 10 subcategories.


In [32]:
# Cell 6 - Load embedding model (same model used in Step 2)
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"Model loaded: all-MiniLM-L6-v2")

test_vec = model.encode("test sentence")
print(f"Embedding dimension: {len(test_vec)}")

Model loaded: all-MiniLM-L6-v2
Embedding dimension: 384


In [33]:
# Cell 7 - Retrieve Top-10 for all 50 queries and compute metrics
from tqdm import tqdm

K = 10
results_records = []

for q in tqdm(queries, desc="Retrieving"):
    qid       = q["query_id"]
    qtext     = q["query_text"]
    subcat    = q["subcategory"]
    pool_size = max(subcat_pool_size.get(subcat, 0), 1)

    q_vec = model.encode(qtext).tolist()

    chroma_res = collection.query(
        query_embeddings=[q_vec],
        n_results=K,
        include=["metadatas", "distances"]
    )

    paper_ids = chroma_res["ids"][0]
    metadatas = chroma_res["metadatas"][0]
    distances = chroma_res["distances"][0]

    relevance = [
        1 if subcat in meta.get("categories", "") else 0
        for meta in metadatas
    ]

    num_relevant   = sum(relevance)
    precision_at_k = num_relevant / K
    recall_at_k    = num_relevant / pool_size
    hit_rate_at_k  = 1 if num_relevant > 0 else 0

    results_records.append({
        "query_id"            : qid,
        "query_text"          : qtext,
        "subcategory"         : subcat,
        "retrieved_paper_ids" : paper_ids,
        "relevance"           : relevance,
        "precision_at_10"     : round(precision_at_k, 4),
        "recall_at_10"        : round(recall_at_k, 6),
        "hit_rate_at_10"      : hit_rate_at_k,
    })

print(f"\nDone. {len(results_records)} queries evaluated.")

Retrieving: 100%|██████████| 50/50 [00:00<00:00, 85.76it/s]


Done. 50 queries evaluated.


In [34]:
# Cell 8 - Save results
import json

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(OUT_PATH, "w") as f:
    json.dump(results_records, f, indent=2)

print(f"Saved {len(results_records)} records to: {OUT_PATH}")

Saved 50 records to: /Users/jicijiang/neu/cs6200/project/FairSearch-qBio/data/processed/retrieval_results.json


In [35]:
# Cell 9 - Overall summary
import statistics

p_vals  = [r["precision_at_10"] for r in results_records]
rc_vals = [r["recall_at_10"]    for r in results_records]
hr_vals = [r["hit_rate_at_10"]  for r in results_records]

print("=" * 50)
print("OVERALL BASELINE METRICS (K=10, n=50 queries)")
print("=" * 50)
print(f"  Mean Precision@10 : {statistics.mean(p_vals):.4f}")
print(f"  Mean Recall@10    : {statistics.mean(rc_vals):.6f}  (low — large relevant pool)")
print(f"  Mean HitRate@10   : {statistics.mean(hr_vals):.4f}")
print(f"  HitRate=1 queries : {sum(hr_vals)}/{len(hr_vals)}")

OVERALL BASELINE METRICS (K=10, n=50 queries)
  Mean Precision@10 : 0.6540
  Mean Recall@10    : 0.001387  (low — large relevant pool)
  Mean HitRate@10   : 0.9600
  HitRate=1 queries : 48/50


In [36]:
# Cell 10 - Per-subcategory breakdown
from collections import defaultdict

subcat_metrics = defaultdict(lambda: {"precision": [], "recall": [], "hit_rate": []})

for r in results_records:
    sc = r["subcategory"]
    subcat_metrics[sc]["precision"].append(r["precision_at_10"])
    subcat_metrics[sc]["recall"].append(r["recall_at_10"])
    subcat_metrics[sc]["hit_rate"].append(r["hit_rate_at_10"])

print(f"{'Subcategory':<14} {'Queries':>7} {'Pool':>7} {'Prec@10':>9} {'Rec@10':>10} {'Hit@10':>8}")
print("-" * 60)

for sc in sorted(subcat_metrics):
    m      = subcat_metrics[sc]
    n      = len(m["precision"])
    avg_p  = statistics.mean(m["precision"])
    avg_r  = statistics.mean(m["recall"])
    avg_hr = statistics.mean(m["hit_rate"])
    pool   = subcat_pool_size.get(sc, 0)
    print(f"{sc:<14} {n:>7} {pool:>7,} {avg_p:>9.4f} {avg_r:>10.6f} {avg_hr:>8.4f}")

Subcategory    Queries    Pool   Prec@10     Rec@10   Hit@10
------------------------------------------------------------
q-bio.BM             5   6,745    0.7000   0.001038   1.0000
q-bio.CB             4   2,451    0.7250   0.002958   1.0000
q-bio.GN             4   3,835    0.7750   0.002021   1.0000
q-bio.MN             4   4,157    0.8250   0.001984   1.0000
q-bio.NC             6  12,131    0.9667   0.000797   1.0000
q-bio.OT             4   1,586    0.0750   0.000473   0.5000
q-bio.PE             7  12,998    0.9286   0.000714   1.0000
q-bio.QM             7  13,180    0.3429   0.000260   1.0000
q-bio.SC             5   1,808    0.4800   0.002655   1.0000
q-bio.TO             4   2,618    0.6250   0.002388   1.0000


In [37]:
# Cell 11 - Per-query detail (sorted by Precision@10 descending)
sorted_results = sorted(results_records, key=lambda r: r["precision_at_10"], reverse=True)

print(f"{'QID':<6} {'Subcat':<12} {'Prec@10':>9} {'Hit@10':>8}  Query (truncated)")
print("-" * 90)
for r in sorted_results:
    print(
        f"{r['query_id']:<6} "
        f"{r['subcategory']:<12} "
        f"{r['precision_at_10']:>9.4f} "
        f"{r['hit_rate_at_10']:>8}  "
        f"{r['query_text'][:55]}"
    )

QID    Subcat         Prec@10   Hit@10  Query (truncated)
------------------------------------------------------------------------------------------
q008   q-bio.PE        1.0000        1  How do epidemiological models predict the spread of inf
q009   q-bio.PE        1.0000        1  What evolutionary mechanisms drive the emergence of ant
q011   q-bio.PE        1.0000        1  What mathematical models describe predator-prey dynamic
q012   q-bio.PE        1.0000        1  How do game-theoretic models explain the evolution of c
q015   q-bio.NC        1.0000        1  How do neural oscillations contribute to memory consoli
q016   q-bio.NC        1.0000        1  What computational models explain the role of dopamine 
q017   q-bio.NC        1.0000        1  How do sensory neurons encode stimulus intensity throug
q020   q-bio.NC        1.0000        1  What are the computational principles underlying attent
q024   q-bio.BM        1.0000        1  What computational approaches predict RNA s

In [38]:
# Cell 12 - Spot check: inspect Top-10 results for a specific query
INSPECT_QID = "q015"   # change to any query_id

target   = next(r for r in results_records if r["query_id"] == INSPECT_QID)
q_subcat = target["subcategory"]

print(f"Query      : {target['query_text']}")
print(f"Subcategory: {q_subcat}")
print(f"Precision@10: {target['precision_at_10']}   HitRate@10: {target['hit_rate_at_10']}")
print()

q_vec = model.encode(target["query_text"]).tolist()
chroma_res = collection.query(
    query_embeddings=[q_vec],
    n_results=K,
    include=["metadatas", "distances"]
)

print(f"{'#':<3} {'Rel':<4} {'Dist':>6}  {'Categories':<35} Title")
print("-" * 100)
for rank, (pid, meta, dist, rel) in enumerate(
    zip(
        chroma_res["ids"][0],
        chroma_res["metadatas"][0],
        chroma_res["distances"][0],
        target["relevance"]
    ), start=1
):
    marker = "v" if rel else " "
    cats   = meta.get("categories", "")[:34]
    title  = meta.get("title", "")[:55]
    print(f"{rank:<3} [{marker}]  {dist:>6.4f}  {cats:<35} {title}")

Query      : How do neural oscillations contribute to memory consolidation during sleep?
Subcategory: q-bio.NC
Precision@10: 1.0   HitRate@10: 1

#   Rel    Dist  Categories                          Title
----------------------------------------------------------------------------------------------------
1   [v]  0.3290  q-bio.NC                            Computational role of sleep in memory reorganization
2   [v]  0.3611  q-bio.NC                            Querying hippocampal replay with subcortical inputs
3   [v]  0.3657  q-bio.NC                            Structural network heterogeneities and network dynamics
4   [v]  0.3713  q-bio.NC cs.AI cs.ET                A computational account of dreaming: learning and memor
5   [v]  0.3732  cond-mat.dis-nn nlin.CD physics.po  Dreams, endocannabinoids and itinerant dynamics in neur
6   [v]  0.3794  q-bio.NC cond-mat.dis-nn            Cholinergic switch between two types of slow waves in c
7   [v]  0.3858  q-bio.NC                      